# Orderbook Store Example

This notebook demonstrates the new data storage infrastructure for OKX orderbook data.


In [1]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, date
from okx.store import OrderbookStore, populate, FEATURES
import polars as pl


## 1. Setup Store


In [2]:
# Initialize the store
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)


## 2. Populate Raw Data

Fetch and store full depth=5 orderbook data. This only fetches missing dates.


In [4]:
# Populate BTC-USD-SWAP data for a few days
populate(
    store,
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 8, 1),
    end=datetime(2025, 9, 1),
    verbose=True
)


Fetching BTC-USD/SWAP: 3 missing dates (from 31 requested)


  2025-08-07: No data returned from API
  2025-08-09: No data returned from API
✓ Stored 1/3 days as raw orderbook data
  Failed dates: [datetime.date(2025, 8, 7), datetime.date(2025, 8, 9)]


## 3. Example Queries

### 3.1 Depth=1, 1-minute bins (with caching)


In [3]:
# Get depth=1, 1-min binned data (cached as 'd1_1m')
lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=1,
    binning='1m',
    cache_name='d1_1m'
)

# Collect and display
df = lf.collect()
print(f"Shape: {df.shape}")
df.head()


Shape: (5700, 10)


symbol,timeMs,exchTimeMs,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,time_bin
str,i64,i64,f64,f64,i64,f64,f64,i64,datetime[ms]
"""BTC-USD-SWAP.OK""",1756684859827,1756684859826,108213.9,4199.0,26,108214.0,1933.0,6,2025-09-01 00:01:00
"""BTC-USD-SWAP.OK""",1756684919998,1756684919996,108288.1,5163.0,29,108288.2,2724.0,14,2025-09-01 00:02:00
"""BTC-USD-SWAP.OK""",1756684979998,1756684979996,108207.1,4773.0,29,108207.2,4705.0,15,2025-09-01 00:03:00
"""BTC-USD-SWAP.OK""",1756685039857,1756685039856,108180.0,3972.0,26,108180.1,4561.0,17,2025-09-01 00:04:00
"""BTC-USD-SWAP.OK""",1756685099998,1756685099996,108052.9,4313.0,28,108053.0,3569.0,11,2025-09-01 00:05:00


### 3.2 Depth=1, 5-minute bins with features (with caching)


In [4]:
# Get depth=1, 5-min bins with mid, spread, and imbalance features
lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=1,
    binning='5m',
    features=['mid', 'spread', 'rel_spread', 'imbalance1'],
    cache_name='d1_5m_features'
)

df = lf.collect()
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head()


Shape: (1140, 14)
Columns: ['symbol', 'timeMs', 'exchTimeMs', 'bid_1_px', 'bid_1_qty', 'bid_1_ordCnt', 'ask_1_px', 'ask_1_qty', 'ask_1_ordCnt', 'mid', 'spread', 'rel_spread', 'imbalance1', 'time_bin']


symbol,timeMs,exchTimeMs,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,mid,spread,rel_spread,imbalance1,time_bin
str,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,f64,f64,datetime[ms]
"""BTC-USD-SWAP.OK""",1756685099998,1756685099996,108052.9,4313.0,28,108053.0,3569.0,11,108052.95,0.1,9.2547e-7,0.094392,2025-09-01 00:05:00
"""BTC-USD-SWAP.OK""",1756685399987,1756685399985,107941.7,143.0,7,107941.8,4960.0,28,107941.75,0.1,9.2643e-7,-0.943955,2025-09-01 00:10:00
"""BTC-USD-SWAP.OK""",1756685699986,1756685699985,107924.0,989.0,12,107924.1,6369.0,28,107924.05,0.1,9.2658e-7,-0.731177,2025-09-01 00:15:00
"""BTC-USD-SWAP.OK""",1756685999986,1756685999985,107898.0,3893.0,20,107898.1,368.0,5,107898.05,0.1,9.2680e-7,0.827271,2025-09-01 00:20:00
"""BTC-USD-SWAP.OK""",1756686299997,1756686299995,107697.7,2922.0,36,107697.8,170.0,2,107697.75,0.1,9.2852e-7,0.890039,2025-09-01 00:25:00


### 3.3 Custom features with lambda function


In [4]:
# Custom feature: compute volatility over rolling window
def add_volatility(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns([
        pl.col('mid').rolling_std(window_size=10).alias('mid_vol_10')
    ])

# Combine registry features with custom function
# Applying bin and trim first - not default behavoir but desired for volatility.
lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=1,
    binning='1m',
    features=['bin', 'trim','mid', 'spread', add_volatility],
    cache_name='d1_1m_vol'
)

df = lf.collect()
print(f"Columns: {df.columns}")
df.select(['timeMs', 'symbol', 'mid', 'spread', 'mid_vol_10']).head(15)


Columns: ['symbol', 'timeMs', 'exchTimeMs', 'bid_1_px', 'bid_1_qty', 'bid_1_ordCnt', 'ask_1_px', 'ask_1_qty', 'ask_1_ordCnt', 'time_bin', 'mid', 'spread', 'mid_vol_10']


timeMs,symbol,mid,spread,mid_vol_10
i64,str,f64,f64,f64
1756684859827,"""BTC-USD-SWAP.OK""",108213.95,0.1,null
1756684919998,"""BTC-USD-SWAP.OK""",108288.15,0.1,null
1756684979998,"""BTC-USD-SWAP.OK""",108207.15,0.1,null
1756685039857,"""BTC-USD-SWAP.OK""",108180.05,0.1,null
1756685099998,"""BTC-USD-SWAP.OK""",108052.95,0.1,null
…,…,…,…,…
1756685459997,"""BTC-USD-SWAP.OK""",107963.85,0.1,113.44479
1756685519998,"""BTC-USD-SWAP.OK""",107972.15,0.1,91.033061
1756685579998,"""BTC-USD-SWAP.OK""",107965.95,0.1,75.051624


### 3.4 Ad-hoc query (no caching)


In [5]:
# Quick query without caching - just compute on the fly
lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=2,
    binning='10m'
)

df = lf.collect()
print(f"Shape: {df.shape}")
df.head()


Shape: (570, 16)


symbol,timeMs,exchTimeMs,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,bid_2_px,bid_2_qty,bid_2_ordCnt,ask_2_px,ask_2_qty,ask_2_ordCnt,time_bin
str,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,datetime[ms]
"""BTC-USD-SWAP.OK""",1756685399987,1756685399985,107941.7,143.0,7,107941.8,4960.0,28,107939.1,2.0,1,107941.9,128.0,2,2025-09-01 00:10:00
"""BTC-USD-SWAP.OK""",1756685999986,1756685999985,107898.0,3893.0,20,107898.1,368.0,5,107896.0,48.0,1,107901.9,15.0,1,2025-09-01 00:20:00
"""BTC-USD-SWAP.OK""",1756686599996,1756686599995,107766.0,1702.0,12,107766.1,2075.0,19,107764.3,2.0,1,107768.7,207.0,1,2025-09-01 00:30:00
"""BTC-USD-SWAP.OK""",1756687199946,1756687199945,107713.9,3390.0,34,107714.0,89.0,2,107713.8,15.0,1,107716.0,48.0,1,2025-09-01 00:40:00
"""BTC-USD-SWAP.OK""",1756687799997,1756687799995,108127.9,4730.0,29,108128.0,593.0,4,108126.0,48.0,1,108128.9,250.0,5,2025-09-01 00:50:00


## 4. Cache Management


In [6]:
# List all available caches
print("Available caches:")
for cache in store.manifest.list_caches():
    print(f"  - {cache}")


Available caches:
  - d1_1m
  - d1_1m_vol
  - d1_5m_features


In [7]:
# Clear a specific cache
store.clear_cache('d1_1m')


Cleared cache: d1_1m


In [10]:
# Clear all caches (keeps raw data intact)
store.clear_cache()


Cleared all caches


## 5. Integration with Existing Code

You can use the store in place of `fetch_market_data` + `bin_orderbook`:


In [9]:
# Old way:
# swap_df = fetch_market_data(
#     '6', 'SWAP', 'BTC-USD', date, day_end, 'daily',
#     verbose=True, depth=1,
#     include_criterion=lambda fn: fn.startswith('BTC-USD-SWAP'),
#     process_fn=lambda df: bin_orderbook(df, '5min')
# )

# New way (after populating):
swap_lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=1,
    binning='5m',
    cache_name='swap_5m'
)

# Convert to pandas if needed for existing code
swap_df = swap_lf.collect().to_pandas()
print(f"Shape: {swap_df.shape}")
swap_df.head()


Shape: (1140, 10)


,symbol,timeMs,exchTimeMs,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,time_bin
0,BTC-USD-SWAP.OK,1756685099998,1756685099996,108052.9,4313.0,28,108053.0,3569.0,11,2025-09-01 00:05:00
1,BTC-USD-SWAP.OK,1756685399987,1756685399985,107941.7,143.0,7,107941.8,4960.0,28,2025-09-01 00:10:00
2,BTC-USD-SWAP.OK,1756685699986,1756685699985,107924.0,989.0,12,107924.1,6369.0,28,2025-09-01 00:15:00
3,BTC-USD-SWAP.OK,1756685999986,1756685999985,107898.0,3893.0,20,107898.1,368.0,5,2025-09-01 00:20:00
4,BTC-USD-SWAP.OK,1756686299997,1756686299995,107697.7,2922.0,36,107697.8,170.0,2,2025-09-01 00:25:00
